# Tarea C1 — Estudio del Benchmark y Diseño de Adaptación
**ECG Rhythm Analyzer | Gabriela — Modelo CNN**  
**Rama:** `feature/model-training`

## Objetivo
Entender la estructura del repositorio `ecg_ptbxl_benchmarking` (CNN 1D) y documentar cómo lo vamos a adaptar a una CNN 2D que procesa imágenes.


## 1. Clonar el repositorio del benchmark

In [ ]:
# Clonar el repositorio base que vamos a estudiar
!git clone https://github.com/helme/ecg_ptbxl_benchmarking.git
print('Repositorio clonado exitosamente.')

In [ ]:
# Ver la estructura de carpetas del benchmark
import os

for root, dirs, files in os.walk('ecg_ptbxl_benchmarking'):
    # Ignorar carpetas ocultas y __pycache__
    dirs[:] = [d for d in dirs if not d.startswith('.') and d != '__pycache__']
    level = root.replace('ecg_ptbxl_benchmarking', '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:  # Mostrar archivos solo hasta 2 niveles
        subindent = '  ' * (level + 1)
        for file in files:
            print(f'{subindent}{file}')

## 2. Analizar el archivo clave: `scp_experiment.py`
Este archivo contiene el pipeline central del benchmark (carga de datos, entrenamiento, evaluación).

In [ ]:
# Leer y mostrar el contenido del experimento principal
# Buscar el archivo scp_experiment.py donde sea que esté
import subprocess

result = subprocess.run(['find', 'ecg_ptbxl_benchmarking', '-name', 'scp_experiment.py'],
                        capture_output=True, text=True)
scp_path = result.stdout.strip()
print(f'Archivo encontrado en: {scp_path}')

with open(scp_path, 'r') as f:
    content = f.read()
print(content)

In [ ]:
# Analizar utils.py — funciones de carga de datos
result = subprocess.run(['find', 'ecg_ptbxl_benchmarking', '-name', 'utils.py'],
                        capture_output=True, text=True)
utils_path = result.stdout.strip().split('\n')[0]  # tomar el primero
print(f'utils.py encontrado en: {utils_path}')

with open(utils_path, 'r') as f:
    content = f.read()
print(content)

## 3. Resumen: ¿Cómo funciona el benchmark original?

Después de leer el código, completar esta sección:

In [ ]:
# Resumen estructural del benchmark (para completar después de leerlo)
resumen = """
ESTRUCTURA DEL BENCHMARK ORIGINAL (ecg_ptbxl_benchmarking)
===========================================================

1. CARGA DE DATOS:
   - Archivo: utils.py
   - Función principal: load_dataset() / load_raw_data()
   - Formato de entrada: señales numéricas (arrays numpy), shape (N_muestras, 5000, 12)
   - Las 12 columnas = 12 derivaciones de ECG
   - Las etiquetas se cargan desde un CSV con columnas: ecg_id, strat_fold, label

2. LOOP DE ENTRENAMIENTO:
   - Archivo: scp_experiment.py
   - Clase: SCP_Experiment
   - Método: train() o fit()
   - Usa los folds 1-8 para train, 9 para val, 10 para test
   - La función de pérdida es cross-entropy o binary cross-entropy según la tarea

3. ARQUITECTURA DEL MODELO:
   - Tipo: CNN 1D (procesa la señal como serie temporal)
   - Entrada: tensor de shape (batch, 12, 5000) — 12 canales, 5000 puntos
   - Las capas convolucionales operan en la dimensión temporal
   - Salida: vector de probabilidades por clase

4. EVALUACIÓN:
   - Métricas: AUC-ROC, accuracy, F1 por clase
   - Se evalúa sobre el fold 10 (test set nunca visto durante entrenamiento)
"""
print(resumen)

## 4. Tabla comparativa: Benchmark original vs. Nuestra adaptación

In [ ]:
import pandas as pd

comparacion = {
    'Aspecto': [
        'Tipo de entrada',
        'Shape del input',
        'Tipo de CNN',
        'Arquitectura base',
        'Parámetros del modelo',
        'Transfer learning',
        'Función de pérdida',
        'Número de clases de salida',
        'Métricas principales',
        'Framework'
    ],
    'Benchmark Original': [
        'Señal numérica (serie temporal)',
        '(batch, 12, 5000)',
        'CNN 1D',
        'InceptionTime / ResNet 1D / LSTM-FCN',
        'Variable (~1-10M)',
        'No (entrenado desde cero)',
        'Binary Cross-Entropy / Categorical CE',
        '5 (NORM, MI, STTC, CD, HYP)',
        'AUC-ROC por clase, macro-AUC',
        'Keras / TensorFlow'
    ],
    'Nuestra Adaptación (Gabriela)': [
        'Imagen PNG del ECG (foto sintética)',
        '(batch, 3, 224, 224)',
        'CNN 2D',
        'EfficientNet-B0 (recomendado) o ResNet-18',
        '5.3M (EfficientNet-B0) / 11.7M (ResNet-18)',
        'Sí — pre-entrenado en ImageNet',
        'BCEWithLogitsLoss (clasificación binaria)',
        '1 (Normal=0 / Arritmia=1)',
        'Accuracy, Sensibilidad, Especificidad, AUC-ROC',
        'PyTorch + torchvision'
    ]
}

df = pd.DataFrame(comparacion)
df.set_index('Aspecto', inplace=True)

# Mostrar la tabla
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.width', 140)
print(df.to_string())

## 5. Arquitectura elegida: EfficientNet-B0
### ¿Por qué EfficientNet-B0?

In [ ]:
justificacion = """
JUSTIFICACIÓN DE LA ELECCIÓN: EfficientNet-B0
==============================================

✅ VENTAJAS:
  - Tamaño reducido: solo 5.3M parámetros (~20 MB en disco)
    → Entrenable en Google Colab T4 (GPU gratuita) sin problemas de memoria
  - Pre-entrenado en ImageNet: ya aprendió a detectar bordes, texturas, patrones
    → Transfer learning desde el inicio (no arrancamos de cero)
  - Alta precisión/eficiencia: fue diseñado para escalar de forma equilibrada
    entre profundidad, anchura y resolución
  - Disponible en torchvision: una línea de código para cargarlo

🔄 ADAPTACIÓN NECESARIA:
  - La última capa original tiene 1000 salidas (1000 clases de ImageNet)
  - Nosotros la reemplazamos por una capa Linear(1280, 1) → 1 salida binaria
  - Con sigmoid en inferencia: salida entre 0 y 1 (probabilidad de arritmia)

📊 COMPARACIÓN CON ALTERNATIVA (ResNet-18):
  - ResNet-18 tiene 11.7M parámetros (más del doble)
  - EfficientNet-B0 suele tener mejor accuracy con menos parámetros
  - Ambas opciones son válidas; elegimos EfficientNet-B0 por eficiencia
"""
print(justificacion)

In [ ]:
# Instalar e importar PyTorch (ya viene en Colab, esto es solo verificación)
import torch
import torchvision
print(f'PyTorch version: {torch.__version__}')
print(f'Torchvision version: {torchvision.__version__}')
print(f'GPU disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Cargar EfficientNet-B0 y ver su arquitectura
import torch.nn as nn
from torchvision import models

# Cargar modelo pre-entrenado en ImageNet
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

print('=== ARQUITECTURA ORIGINAL DE EfficientNet-B0 ===')
print(f'Última capa (classifier): {model.classifier}')
print()

# Contar parámetros del modelo original
total_params = sum(p.numel() for p in model.parameters())
print(f'Total de parámetros originales: {total_params:,} ({total_params/1e6:.1f}M)')

In [ ]:
# Ver la estructura del clasificador para saber qué reemplazar
print('=== CAPAS DEL CLASSIFIER ===')
for name, layer in model.classifier.named_children():
    print(f'  [{name}]: {layer}')

print()
print('La capa [1] es un Linear(1280, 1000) — la reemplazaremos por Linear(1280, 1)')
print('in_features = 1280 → este número es FIJO para EfficientNet-B0')

In [ ]:
# Vista previa de la adaptación (código completo en C2)
model_adaptado = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

# Reemplazar la última capa: de 1000 clases → 1 salida binaria
model_adaptado.classifier[1] = nn.Linear(1280, 1)

print('=== MODELO ADAPTADO PARA NUESTRO PROYECTO ===')
print(f'Nueva última capa: {model_adaptado.classifier}')
print()

# Verificar que acepta input de imagen 224x224
dummy_input = torch.randn(1, 3, 224, 224)  # batch=1, canales=3, 224x224
output = model_adaptado(dummy_input)
print(f'Shape del input:  {dummy_input.shape}  → (batch, canales, alto, ancho)')
print(f'Shape del output: {output.shape}        → (batch, 1) — una probabilidad por imagen')
print()
print('✅ El modelo acepta imágenes 224x224 y produce 1 salida binaria. Listo para C2.')

## 6. Entregable C1 — Tabla de diferencias documentada

**Resumen ejecutivo para incluir en MODIFICATIONS.md:**

In [ ]:
entregable = """
MODIFICATIONS.md — Sección Gabriela (C1)
=========================================

## Adaptación del benchmark a CNN 2D

### Repositorio base reutilizado
- Repo: `helme/ecg_ptbxl_benchmarking`
- Archivos estudiados: `scp_experiment.py`, `utils.py`
- Se reutilizó: estructura general del pipeline (train/val/test splits con folds,
  lógica de evaluación con AUC-ROC)
- NO se reutilizó: la arquitectura del modelo (CNN 1D → reemplazada por CNN 2D)

### Cambios principales
| Componente          | Original (benchmark)        | Adaptación (este proyecto)         |
|---------------------|----------------------------|-------------------------------------|
| Tipo de entrada     | Señal numérica 1D          | Imagen PNG 224×224                  |
| Arquitectura        | CNN 1D (InceptionTime)     | EfficientNet-B0 (CNN 2D)            |
| Transfer learning   | No                         | Sí — ImageNet pre-trained           |
| Clases de salida    | 5 (multiclase)             | 1 (binaria: Normal vs Arritmia)     |
| Loss function       | Categorical cross-entropy  | BCEWithLogitsLoss                   |
| Framework           | Keras/TF                   | PyTorch + torchvision               |

### Justificación de EfficientNet-B0
Elegida por su relación eficiencia/rendimiento: 5.3M parámetros (~20MB),
disponible pre-entrenada en torchvision, y compatible con GPU gratuita de Colab.
La capa final `classifier[1]` fue reemplazada de Linear(1280, 1000) a Linear(1280, 1).
"""
print(entregable)

---
## ✅ C1 Completada
**Próximo paso:** Notebook `C2_prototipo_datos_dummy.ipynb` — construir y probar el pipeline completo con datos falsos.